In [1]:
from pathlib import Path
import pandas as pd
import pyreadstat

ROOT = Path("../")
W1_INHOME = ROOT / "data" / "W1" / "w1inhome.sas7bdat"
W1_CONTEXT = ROOT / "data" / "W1" / "w1context.sas7bdat"
OUT_CSV = ROOT / "outputs" / "w1_confounder_candidates.csv"

CATEGORIES = {
    "Academic/Cognitive": ["grade", "school", "learn", "special education", "repeat"],
    "Health/Physical": ["health", "weight", "height", "illness", "disability", "sleep", "doctor"],
    "Health/Mental": ["depress", "sad", "happy", "esteem", "cry", "counsel", "psych"],
    "Risk Behaviors": ["smoke", "cigarette", "alcohol", "drink", "drug", "marijuana", "arrest", "trouble", "police"],
    "SES/Financial": ["income", "educat", "welfare", "pay", "money", "afford", "food stamp"],
    "Family/Home": ["parent", "mother", "father", "live with", "born", "sibling"],
    "Demographics": ["race", "hispanic", "age", "sex", "gender"],
}

def search_labels(path: Path, source_name: str) -> list:
    print(f"Reading metadata for {source_name}...")
    _, meta = pyreadstat.read_sas7bdat(str(path), metadataonly=True)
    labels = meta.column_names_to_labels
    
    hits = []
    for var, label in labels.items():
        if not label: continue
        label_lower = label.lower()
        
        # Check against categories
        matched_cats = []
        for cat, keywords in CATEGORIES.items():
            if any(kw in label_lower for kw in keywords):
                matched_cats.append(cat)
                
        if matched_cats:
            hits.append({
                "Source": source_name,
                "Variable": var,
                "Label": label,
                "Categories": ", ".join(matched_cats)
            })
    return hits

hits = []
if W1_INHOME.exists():
    hits.extend(search_labels(W1_INHOME, "w1_inhome"))
if W1_CONTEXT.exists():
    hits.extend(search_labels(W1_CONTEXT, "w1_context"))
        
df = pd.DataFrame(hits)
df.to_csv(OUT_CSV, index=False)
print(f"Found {len(df)} candidate confounders! Saved to {OUT_CSV.name}")


Reading metadata for w1_inhome...
Reading metadata for w1_context...
Found 862 candidate confounders! Saved to w1_confounder_candidates.csv


In [2]:
import pandas as pd
from pathlib import Path

ROOT = Path("../")
IN_CSV = ROOT / "outputs" / "w1_confounder_candidates.csv"
OUT_CSV = ROOT / "outputs" / "w1_final_confounders.csv"

FINAL_CONFOUNDERS = [
    # Demographics
    "BIO_SEX", "PA2", "H1GI8", "H1GI11", 
    # SES & Household
    "PA55", "PA12", "PA57D",
    # Baseline Cognitive & Academic
    "AH_PVT", "H1ED5", "PC38", "PC39",
    # Baseline Health & Risk (Keep it binary/high-level to avoid missingness)
    "H1GH1", "PC53", "H1TO1", "H1TO12",
    # Environment & Contextual Context
    "H1NB5", "BST90P01", "BST90P15"
]

if not IN_CSV.exists():
    print(f"Cannot find {IN_CSV}")

# Load the big candidate list
df = pd.read_csv(IN_CSV)
    
# Filter only for our final curated list
final_df = df[df["Variable"].isin(FINAL_CONFOUNDERS)].copy()

# Group them nicely
final_df.sort_values(by=["Source", "Variable"], inplace=True)
    
# Save the polished list
final_df.to_csv(OUT_CSV, index=False)
    
print(f"Successfully extracted {len(final_df)} highly relevant confounders!")
print(f"Saved to: {OUT_CSV.name}")
    
# Print them out to the console for a quick sanity check
print("\n--- Final Confounder List ---")
for _, row in final_df.iterrows():
    print(f"[{row['Source']}] {row['Variable']}: {row['Label']}")


Successfully extracted 16 highly relevant confounders!
Saved to: w1_final_confounders.csv

--- Final Confounder List ---
[w1_context] BST90P15: Median Household Income
[w1_inhome] AH_PVT: ADD HEALTH PICTURE VOCABULARY TEST SCORE
[w1_inhome] BIO_SEX: BIOLOGICAL SEX-W1
[w1_inhome] H1ED5: S5Q5 HAVE YOU EVER REPEATED A GRADE-W1
[w1_inhome] H1GH1: S3Q1 GENERAL HEALTH-W1
[w1_inhome] H1GI11: S1Q11 BORN IN THE UNITED STATES-W1
[w1_inhome] H1GI8: S1Q8 RACE-SINGLE CATEGORY-W1
[w1_inhome] H1TO1: S28Q1 EVER SMOKED A CIGARETTE-W1
[w1_inhome] H1TO12: S280Q12 DRINK ALCOHOL > 2-3 TIMES-W1
[w1_inhome] PA12: A12 LEVEL OF EDUCATION-PQ
[w1_inhome] PA2: A2 AGE OF RESPONDENT-PQ
[w1_inhome] PA55: A55 TOTAL HOUSEHOLD INCOME-PQ
[w1_inhome] PA57D: A57D RECEIVE FOOD STAMPS-PQ
[w1_inhome] PC38: C38 LEARNING DISABILITY-PQ
[w1_inhome] PC39: C39 SPECIAL EDUCATION-PQ
[w1_inhome] PC53: C53 CONSIDER A DISABILITY-PQ


In [3]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

ROOT = Path("../")
W1_NETWORK = ROOT / "data" / "W1" / "w1network.sas7bdat"

print("Loading W1 network data...")
df_net, meta_net = pyreadstat.read_sas7bdat(str(W1_NETWORK))

net_vars = ['AID', 'IDGX2', 'ODGX2', 'BCENT10X']
df_treatment = df_net[net_vars].copy()

for col in ['IDGX2', 'ODGX2', 'BCENT10X']:
    df_treatment[col] = pd.to_numeric(df_treatment[col], errors='coerce')

# Keep network-missing respondents as NaN treatment to avoid misclassifying them as controls.
observed_network = df_treatment[['IDGX2', 'ODGX2', 'BCENT10X']].notna().all(axis=1)
df_treatment['T_Network_Observed'] = observed_network.astype(int)

df_treatment['T_Strict_Isolation'] = np.nan
df_treatment['T_Zero_InDegree'] = np.nan
df_treatment['T_High_Centrality'] = np.nan

df_treatment.loc[observed_network, 'T_Strict_Isolation'] = np.where(
    (df_treatment.loc[observed_network, 'IDGX2'] == 0) & (df_treatment.loc[observed_network, 'ODGX2'] == 0), 1, 0
)
df_treatment.loc[observed_network, 'T_Zero_InDegree'] = np.where(
    df_treatment.loc[observed_network, 'IDGX2'] == 0, 1, 0
)

# Main treatment split for centrality: above observed-network median (treated) vs lower/equal (control).
med_cent = df_treatment.loc[observed_network, 'BCENT10X'].median()
df_treatment.loc[observed_network, 'T_High_Centrality'] = np.where(
    df_treatment.loc[observed_network, 'BCENT10X'] > med_cent, 1, 0
)

print("\n--- Treatment Group Distributions (Observed network only) ---")
for t_col in ["T_Strict_Isolation", "T_Zero_InDegree", "T_High_Centrality"]:
    counts = df_treatment.loc[observed_network, t_col].value_counts(dropna=False)
    print(f"\n{t_col}:")
    print(f"  0 (Control): {int(counts.get(0.0, 0))}")
    print(f"  1 (Treated): {int(counts.get(1.0, 0))}")

print("\nT_High_Centrality definition: 1 if BCENT10X > observed-network median; 0 otherwise.")
print(f"Observed-network median BCENT10X: {med_cent:.3f}")
print(f"Network exposure observed for {observed_network.sum()} / {len(df_treatment)} respondents.")

Loading W1 network data...

--- Treatment Group Distributions (Observed network only) ---

T_Strict_Isolation:
  0 (Control): 4244
  1 (Treated): 153

T_Zero_InDegree:
  0 (Control): 4020
  1 (Treated): 377

T_High_Centrality:
  0 (Control): 2199
  1 (Treated): 2198

T_High_Centrality definition: 1 if BCENT10X > observed-network median; 0 otherwise.
Observed-network median BCENT10X: 0.725
Network exposure observed for 4397 / 6504 respondents.


In [4]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

ROOT = Path("../")
W5_MAIN = ROOT / "data" / "W5" / "pwave5.xpt"

print("Loading W5 data...")
df_w5, meta_w5 = pyreadstat.read_xport(str(W5_MAIN))

df_outcome = df_w5[['AID', 'MODE']].copy()
df_outcome['W5_ModeEligible'] = df_outcome['MODE'].isin(['I', 'T']).astype(int)

# Valid ranges for word recall are 0 to 15. Reserve/not-asked codes are set to NaN.
for col in ['C5WD90_1', 'C5WD60_1']:
    df_outcome[col] = pd.to_numeric(df_w5[col], errors='coerce')
    df_outcome[col] = df_outcome[col].where((df_outcome[col] >= 0) & (df_outcome[col] <= 15), np.nan)

bds_score = np.zeros(len(df_w5))
any_valid = np.zeros(len(df_w5), dtype=bool)

for L in range(2, 9):
    item_a = pd.to_numeric(df_w5.get(f"H5MH{L+1}A"), errors='coerce')
    item_b = pd.to_numeric(df_w5.get(f"H5MH{L+1}B"), errors='coerce')

    if item_a is not None and item_b is not None:
        valid_a = (item_a == 0) | (item_a == 1)
        valid_b = (item_b == 0) | (item_b == 1)
        any_valid = any_valid | valid_a | valid_b

        success = (item_a == 1) | (item_b == 1)
        bds_score = np.where(success, L, bds_score)

df_outcome['W5_BDS'] = bds_score
df_outcome['W5_BDS'] = df_outcome['W5_BDS'].where(any_valid, np.nan)

# Calculate Z-scores and a composite requiring at least 2 available component tests.
df_outcome['Z_Immediate'] = (df_outcome['C5WD90_1'] - df_outcome['C5WD90_1'].mean()) / df_outcome['C5WD90_1'].std()
df_outcome['Z_Delayed'] = (df_outcome['C5WD60_1'] - df_outcome['C5WD60_1'].mean()) / df_outcome['C5WD60_1'].std()
df_outcome['Z_BDS'] = (df_outcome['W5_BDS'] - df_outcome['W5_BDS'].mean()) / df_outcome['W5_BDS'].std()

z_cols = ['Z_Immediate', 'Z_Delayed', 'Z_BDS']
n_tests = df_outcome[z_cols].notna().sum(axis=1)
df_outcome['W5_Cognitive_Composite'] = df_outcome[z_cols].mean(axis=1, skipna=True)
df_outcome.loc[n_tests < 2, 'W5_Cognitive_Composite'] = np.nan

print("\n--- Outcome Distribution ---")
print(df_outcome[['C5WD90_1', 'C5WD60_1', 'W5_BDS', 'W5_Cognitive_Composite']].describe())

print("\n--- Mode x Cognitive Composite Availability ---")
mode_tab = pd.crosstab(df_outcome['MODE'], df_outcome['W5_Cognitive_Composite'].notna(), margins=True)
print(mode_tab)

Loading W5 data...

--- Outcome Distribution ---
         C5WD90_1    C5WD60_1      W5_BDS  W5_Cognitive_Composite
count  623.000000  620.000000  625.000000              623.000000
mean     6.199037    4.650000    5.142400               -0.001286
std      2.102410    2.049095    1.556792                0.789675
min      1.000000    0.000000    0.000000               -2.519124
25%      5.000000    3.000000    4.000000               -0.540449
50%      6.000000    5.000000    5.000000               -0.009237
75%      7.000000    6.000000    6.000000                0.521975
max     15.000000   12.000000    8.000000                3.202884

--- Mode x Cognitive Composite Availability ---
W5_Cognitive_Composite  False  True   All
MODE                                     
I                         200   523   723
M                         147     0   147
S                           4     0     4
T                           1   100   101
W                        3221     0  3221
All           

In [5]:
import pandas as pd
import pyreadstat
from pathlib import Path

ROOT = Path("../")
W1_INHOME = ROOT / "data" / "W1" / "w1inhome.sas7bdat"
W1_CONTEXT = ROOT / "data" / "W1" / "w1context.sas7bdat"
CANDIDATE_CSV = ROOT / "outputs" / "w1_confounder_candidates.csv"

print("Loading Confounder Data...")
df_inhome, _ = pyreadstat.read_sas7bdat(str(W1_INHOME))
df_context, _ = pyreadstat.read_sas7bdat(str(W1_CONTEXT))

INHOME_CORE_CONFOUNDERS = [
    "BIO_SEX", "PA2", "H1GI8", "H1GI11", "PA55", "PA12", "PA57D",
    "AH_PVT", "H1ED5", "PC38", "PC39", "H1GH1", "PC53", "H1TO1",
    "H1TO12", "H1NB5"
]
CONTEXT_CORE_CONFOUNDERS = ["BST90P01", "BST90P15"]

# Pull extra baseline confounders from the metadata-screened candidate file.
EXTRA_CATEGORY_PRIORITY = [
    "SES/Financial",
    "Family/Home",
    "Health/Mental",
    "Academic/Cognitive",
    "Health/Physical",
    "Demographics",
    "Risk Behaviors",
]
MAX_EXTRA_PER_SOURCE = 12

def unique_preserve(seq):
    return list(dict.fromkeys(seq))

extra_inhome = []
extra_context = []

if CANDIDATE_CSV.exists():
    cand = pd.read_csv(CANDIDATE_CSV)
    cand["Source"] = cand["Source"].astype(str).str.lower()
    cand["Categories"] = cand["Categories"].fillna("").astype(str)

    for source_name, core_vars in [
        ("w1_inhome", INHOME_CORE_CONFOUNDERS),
        ("w1_context", CONTEXT_CORE_CONFOUNDERS),
    ]:
        src = cand[cand["Source"] == source_name].copy()
        src = src[~src["Variable"].isin(core_vars)]
        cat_mask = src["Categories"].apply(
            lambda txt: any(cat in txt for cat in EXTRA_CATEGORY_PRIORITY)
        )
        src = src[cat_mask].drop_duplicates("Variable")
        picked = src["Variable"].head(MAX_EXTRA_PER_SOURCE).tolist()
        if source_name == "w1_inhome":
            extra_inhome = picked
        else:
            extra_context = picked

inhome_selected = unique_preserve(
    INHOME_CORE_CONFOUNDERS + [v for v in extra_inhome if v in df_inhome.columns]
 )
context_selected = unique_preserve(
    CONTEXT_CORE_CONFOUNDERS + [v for v in extra_context if v in df_context.columns]
 )

df_inhome_conf = df_inhome[['AID'] + inhome_selected].copy()
df_context_conf = df_context[['AID'] + context_selected].copy()

def decode_aid(series):
    if pd.api.types.is_object_dtype(series):
        return series.apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else str(x))
    return series.astype(str)

print("Decoding AIDs for merging...")
df_treatment['AID'] = decode_aid(df_treatment['AID'])
df_outcome['AID'] = decode_aid(df_outcome['AID'])
df_inhome_conf['AID'] = decode_aid(df_inhome_conf['AID'])
df_context_conf['AID'] = decode_aid(df_context_conf['AID'])

print("Merging dataframes...")
causal_df = df_treatment[['AID', 'T_Network_Observed', 'T_Strict_Isolation', 'T_Zero_InDegree', 'T_High_Centrality']].copy()

causal_df = pd.merge(
    causal_df,
    df_outcome[['AID', 'MODE', 'W5_ModeEligible', 'C5WD90_1', 'C5WD60_1', 'W5_BDS', 'W5_Cognitive_Composite']],
    on='AID',
    how='inner'
 )
causal_df = pd.merge(causal_df, df_inhome_conf, on='AID', how='inner')
causal_df = pd.merge(causal_df, df_context_conf, on='AID', how='left')

print(f"\nFinal Merged DataFrame Shape: {causal_df.shape}")
print(f"Number of W1 respondents successfully matched to W5: {len(causal_df)}")
print(f"Core in-home confounders kept: {len(INHOME_CORE_CONFOUNDERS)}")
print(f"Extra in-home confounders kept: {len(inhome_selected) - len(INHOME_CORE_CONFOUNDERS)}")
print(f"Core context confounders kept: {len(CONTEXT_CORE_CONFOUNDERS)}")
print(f"Extra context confounders kept: {len(context_selected) - len(CONTEXT_CORE_CONFOUNDERS)}")
print("\nColumns in `causal_df`:")
print(causal_df.columns.tolist())

causal_df.head()

Loading Confounder Data...
Decoding AIDs for merging...
Merging dataframes...

Final Merged DataFrame Shape: (4196, 52)
Number of W1 respondents successfully matched to W5: 4196
Core in-home confounders kept: 16
Extra in-home confounders kept: 12
Core context confounders kept: 2
Extra context confounders kept: 11

Columns in `causal_df`:
['AID', 'T_Network_Observed', 'T_Strict_Isolation', 'T_Zero_InDegree', 'T_High_Centrality', 'MODE', 'W5_ModeEligible', 'C5WD90_1', 'C5WD60_1', 'W5_BDS', 'W5_Cognitive_Composite', 'BIO_SEX', 'PA2', 'H1GI8', 'H1GI11', 'PA55', 'PA12', 'PA57D', 'AH_PVT', 'H1ED5', 'PC38', 'PC39', 'H1GH1', 'PC53', 'H1TO1', 'H1TO12', 'H1NB5', 'SCH_YR', 'SMP03', 'H1GI3', 'H1GI4', 'H1GI5A', 'H1GI5B', 'H1GI5C', 'H1GI5D', 'H1GI5E', 'H1GI5F', 'H1GI6A', 'H1GI6B', 'BST90P01', 'BST90P15', 'BST90P02', 'BST90P03', 'BST90P04', 'BST90P05', 'BST90P06', 'BST90P07', 'BST90P16', 'BST90P17', 'BST90P18', 'BST90P20', 'BST90P21']


,AID,T_Network_Observed,T_Strict_Isolation,T_Zero_InDegree,T_High_Centrality,MODE,W5_ModeEligible,C5WD90_1,C5WD60_1,W5_BDS,...,BST90P03,BST90P04,BST90P05,BST90P06,BST90P07,BST90P16,BST90P17,BST90P18,BST90P20,BST90P21
0,57101310,0,NaN,NaN,NaN,W,0,NaN,NaN,NaN,...,0.014,1.0,2.0,36.0,0.983,0.938,17000.0,0.947,2.0,0.838
1,57111071,0,NaN,NaN,NaN,W,0,NaN,NaN,NaN,...,0.010,1.0,2.0,35.0,0.980,0.941,38000.0,0.903,2.0,0.919
2,57111786,0,NaN,NaN,NaN,T,1,9.0,8.0,6.0,...,0.382,1.0,2.0,31.0,0.913,0.818,46000.0,0.812,3.0,0.860
3,57113943,0,NaN,NaN,NaN,I,1,7.0,3.0,3.0,...,0.566,1.0,2.0,39.0,0.979,0.931,999998.0,9998.000,2.0,0.948
4,57117997,0,NaN,NaN,NaN,W,0,NaN,NaN,NaN,...,0.808,3.0,2.0,27.0,0.952,0.906,44000.0,0.872,2.0,0.845


In [6]:
# Reserve-code cleanup across numeric columns before modeling.
# This avoids treating survey missing/reserve codes as valid numeric values.
RESERVE_CODES = {995, 996, 997, 998, 999, 9995, 9996, 9997, 9998, 9999}
numeric_cols = causal_df.select_dtypes(include=[np.number]).columns.tolist()

n_replaced_total = 0
for c in numeric_cols:
    n_matches = causal_df[c].isin(RESERVE_CODES).sum()
    if n_matches > 0:
        causal_df.loc[causal_df[c].isin(RESERVE_CODES), c] = np.nan
        n_replaced_total += int(n_matches)

print(f"Reserve-code cleanup complete: replaced {n_replaced_total} coded values across {len(numeric_cols)} numeric columns.")

Reserve-code cleanup complete: replaced 1123 coded values across 50 numeric columns.


In [7]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Target population for primary estimand: respondents with observed network exposure.
# Main exposure contrast: high Bonacich-style centrality (> observed-network median) vs lower/equal centrality.
analytic_df = causal_df[causal_df['T_Network_Observed'] == 1].copy()

# Do NOT restrict by W5 mode eligibility here.
# Treat missing cognitive composite as outcome missingness.
analytic_df['Y_observed'] = analytic_df['W5_Cognitive_Composite'].notna().astype(int)

print(f"Rows in target population (network observed, median split): {len(analytic_df)}")
print(f"Outcome observed rate in target population: {analytic_df['Y_observed'].mean():.3f}")

print("Loading W5 Weights and Clusters...")
W5_WEIGHTS = ROOT / "data" / "W5" / "p5weight.xpt"
df_w5_weights, _ = pyreadstat.read_xport(str(W5_WEIGHTS))
df_w5_weights['AID'] = decode_aid(df_w5_weights['AID'])
analytic_df = pd.merge(
    analytic_df,
    df_w5_weights[['AID', 'GSW5', 'CLUSTER2']],
    on='AID',
    how='inner'
)

# Baseline covariate encoding only (no MODE terms).
categorical_cols = ['H1GI8', 'BST90P01']
for col in categorical_cols:
    if col in analytic_df.columns:
        analytic_df[col] = analytic_df[col].astype('category')
analytic_df = pd.get_dummies(analytic_df, columns=[c for c in categorical_cols if c in analytic_df.columns], drop_first=True)

treatment_cols = ['T_Strict_Isolation', 'T_Zero_InDegree', 'T_High_Centrality']
outcome_cols = ['C5WD90_1', 'C5WD60_1', 'W5_BDS', 'W5_Cognitive_Composite']
meta_cols = ['AID', 'MODE', 'GSW5', 'CLUSTER2', 'W5_ModeEligible', 'T_Network_Observed', 'Y_observed']

# Guardrail: exclude post-treatment MODE dummies if they appear.
post_treatment_cols = [c for c in analytic_df.columns if c.startswith('MODE_')]
confounder_cols = [
    c for c in analytic_df.columns
    if c not in treatment_cols + outcome_cols + meta_cols + post_treatment_cols
]

# Data-adaptive trimming: remove near-empty and near-constant covariates.
missing_rate = analytic_df[confounder_cols].isna().mean()
high_missing = missing_rate[missing_rate > 0.45].index.tolist()
if high_missing:
    confounder_cols = [c for c in confounder_cols if c not in high_missing]
near_constant = [c for c in confounder_cols if analytic_df[c].nunique(dropna=True) <= 1]
if near_constant:
    confounder_cols = [c for c in confounder_cols if c not in near_constant]

print(f"Confounders before trimming: {len(missing_rate)}")
print(f"Dropped for >45% missingness: {len(high_missing)}")
print(f"Dropped as near-constant: {len(near_constant)}")
print(f"Confounders used for modeling: {len(confounder_cols)}")

print("Imputing missing baseline covariate data...")
imputer = SimpleImputer(strategy='median')
analytic_df[confounder_cols] = imputer.fit_transform(analytic_df[confounder_cols])

# Observation model P(R=1 | X, T), where R = Y_observed.
selection_features = confounder_cols + treatment_cols
sel_scaler = StandardScaler()
X_sel_scaled = pd.DataFrame(
    sel_scaler.fit_transform(analytic_df[selection_features]),
    columns=selection_features,
    index=analytic_df.index
)

sel_model = LogisticRegression(max_iter=5000, random_state=42)
sel_model.fit(X_sel_scaled, analytic_df['Y_observed'])

analytic_df['P_OBS_RAW'] = sel_model.predict_proba(X_sel_scaled)[:, 1]
analytic_df['P_OBS'] = np.clip(analytic_df['P_OBS_RAW'], 0.05, 0.95)

# IPCW for observed outcomes only.
analytic_df['SW_OBS'] = np.where(
    analytic_df['Y_observed'] == 1,
    1.0 / analytic_df['P_OBS'],
    np.nan
)

print("\nObservation model diagnostics:")
print(analytic_df['P_OBS'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
frac_obs_clipped = np.mean((analytic_df['P_OBS_RAW'] < 0.05) | (analytic_df['P_OBS_RAW'] > 0.95))
print(f"Fraction of clipped observation probabilities: {frac_obs_clipped:.3f}")

# Keep observed outcomes for AIPW estimation, weighted to represent target population.
analytic_df = analytic_df[analytic_df['Y_observed'] == 1].copy()

Y = analytic_df['W5_Cognitive_Composite']
T = analytic_df['T_High_Centrality'].astype(int)
X = analytic_df[confounder_cols]
W = analytic_df['GSW5'] * analytic_df['SW_OBS']

print(f"\nFinal Design Matrix X shape: {X.shape}")
print("No NaNs remain in X:", not X.isnull().values.any())
print("No NaNs remain in Y:", not Y.isnull().values.any())
print(f"Effective sample size from final weights: {(W.sum() ** 2) / (W.pow(2).sum()):.1f}")

# Weighted standardized mean difference (SMD) diagnostics for T_High_Centrality.
def weighted_mean(x, w):
    return np.sum(w * x) / np.sum(w)

def weighted_var(x, w):
    mu = weighted_mean(x, w)
    return np.sum(w * (x - mu) ** 2) / np.sum(w)

smd_rows = []
for col in confounder_cols:
    x_col = pd.to_numeric(analytic_df[col], errors='coerce')
    mask = x_col.notna()
    t_mask = mask & (T == 1)
    c_mask = mask & (T == 0)
    if t_mask.sum() == 0 or c_mask.sum() == 0:
        continue
    m1 = weighted_mean(x_col[t_mask], W[t_mask])
    m0 = weighted_mean(x_col[c_mask], W[c_mask])
    v1 = weighted_var(x_col[t_mask], W[t_mask])
    v0 = weighted_var(x_col[c_mask], W[c_mask])
    denom = np.sqrt((v1 + v0) / 2.0)
    smd = 0.0 if denom == 0 else (m1 - m0) / denom
    smd_rows.append((col, abs(smd), smd))

smd_df = pd.DataFrame(smd_rows, columns=['covariate', 'abs_smd', 'smd']).sort_values('abs_smd', ascending=False)
print("\nTop 10 weighted SMD diagnostics (target |SMD| < 0.1):")
print(smd_df.head(10).to_string(index=False))
print(f"Max |SMD| across covariates: {smd_df['abs_smd'].max():.3f}")
print("Ready for Estimation!")

Rows in target population (network observed, median split): 2936
Outcome observed rate in target population: 0.134
Loading W5 Weights and Clusters...
Confounders before trimming: 49
Dropped for >45% missingness: 0
Dropped as near-constant: 0
Confounders used for modeling: 49
Imputing missing baseline covariate data...

Observation model diagnostics:
count    2936.000000
mean        0.134463
std         0.047531
min         0.050000
1%          0.050000
5%          0.073325
50%         0.127480
95%         0.216521
99%         0.289236
max         0.950000
Name: P_OBS, dtype: float64
Fraction of clipped observation probabilities: 0.012

Final Design Matrix X shape: (394, 49)
No NaNs remain in X: True
No NaNs remain in Y: True
Effective sample size from final weights: 239.6

Top 10 weighted SMD diagnostics (target |SMD| < 0.1):
   covariate  abs_smd       smd
    BST90P20 0.347152  0.347152
    BST90P15 0.335025  0.335025
       H1ED5 0.293494 -0.293494
      AH_PVT 0.280158  0.280158
  

In [8]:
analytic_df

,AID,T_Network_Observed,T_Strict_Isolation,T_Zero_InDegree,T_High_Centrality,MODE,W5_ModeEligible,C5WD90_1,C5WD60_1,W5_BDS,...,H1GI8_5.0,H1GI8_6.0,H1GI8_7.0,H1GI8_8.0,H1GI8_9.0,BST90P01_2.0,BST90P01_9.0,P_OBS_RAW,P_OBS,SW_OBS
13,90506849,1,0.0,0.0,1.0,I,1,7.0,5.0,6.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.114975,0.114975,8.697557
19,90540850,1,1.0,1.0,0.0,I,1,5.0,6.0,3.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.276733,0.276733,3.613596
20,90544989,1,0.0,0.0,1.0,I,1,5.0,4.0,3.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.090772,0.090772,11.016670
26,90570373,1,0.0,0.0,1.0,I,1,6.0,7.0,3.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.145685,0.145685,6.864126
31,90570720,1,0.0,0.0,1.0,I,1,5.0,5.0,7.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.155617,0.155617,6.426027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2887,99712907,1,0.0,0.0,1.0,I,1,4.0,3.0,6.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.120661,0.120661,8.287705
2911,99716352,1,0.0,0.0,0.0,T,1,6.0,4.0,5.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.137484,0.137484,7.273600
2921,99717919,1,0.0,0.0,0.0,T,1,7.0,5.0,6.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.132423,0.132423,7.551586
2922,99718007,1,0.0,0.0,0.0,I,1,12.0,11.0,6.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.149976,0.149976,6.667717


In [9]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
import numpy as np
import pandas as pd

def fit_cross_fitted_aipw(
    df,
    confounder_cols,
    treatment_col,
    outcome_col,
    weight_col,
    n_splits=5,
    random_state=42,
    clip=0.05,
):
    """Cross-fitted AIPW for a binary treatment with survey/observation weights."""
    X_local = df[confounder_cols].to_numpy()
    T_local = df[treatment_col].astype(int).to_numpy()
    Y_local = df[outcome_col].to_numpy()
    W_local = df[weight_col].to_numpy()

    n = len(df)
    e_hat = np.full(n, np.nan)
    m1_hat = np.full(n, np.nan)
    m0_hat = np.full(n, np.nan)

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for train_idx, test_idx in splitter.split(X_local, T_local):
        X_train = X_local[train_idx]
        X_test = X_local[test_idx]
        T_train = T_local[train_idx]
        Y_train = Y_local[train_idx]
        W_train = W_local[train_idx]

        if np.unique(T_train).size < 2:
            continue

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Propensity model P(T=1|X) trained on in-fold observations only.
        ps_model = LogisticRegression(max_iter=3000, random_state=random_state)
        ps_model.fit(X_train_scaled, T_train, sample_weight=W_train)
        e_hat[test_idx] = np.clip(ps_model.predict_proba(X_test_scaled)[:, 1], clip, 1 - clip)

        # Outcome models E[Y|T=1,X] and E[Y|T=0,X] using train fold only.
        t1_mask = T_train == 1
        t0_mask = T_train == 0
        if t1_mask.sum() < 5 or t0_mask.sum() < 5:
            continue

        out_model_1 = LinearRegression().fit(
            X_train_scaled[t1_mask],
            Y_train[t1_mask],
            sample_weight=W_train[t1_mask],
        )
        out_model_0 = LinearRegression().fit(
            X_train_scaled[t0_mask],
            Y_train[t0_mask],
            sample_weight=W_train[t0_mask],
        )

        m1_hat[test_idx] = out_model_1.predict(X_test_scaled)
        m0_hat[test_idx] = out_model_0.predict(X_test_scaled)

    valid_mask = np.isfinite(e_hat) & np.isfinite(m1_hat) & np.isfinite(m0_hat)
    if valid_mask.sum() < int(0.8 * n):
        raise ValueError(
            f"Too few valid cross-fitted predictions ({valid_mask.sum()} of {n}); "
            "reduce n_splits or inspect treatment imbalance."
        )

    T_valid = T_local[valid_mask]
    Y_valid = Y_local[valid_mask]
    W_valid = W_local[valid_mask]
    e_valid = e_hat[valid_mask]
    m1_valid = m1_hat[valid_mask]
    m0_valid = m0_hat[valid_mask]

    dr1_valid = m1_valid + (T_valid * (Y_valid - m1_valid) / e_valid)
    dr0_valid = m0_valid + ((1 - T_valid) * (Y_valid - m0_valid) / (1 - e_valid))
    individual_te_valid = dr1_valid - dr0_valid
    ate_local = np.average(individual_te_valid, weights=W_valid)

    individual_te = pd.Series(np.nan, index=df.index, dtype=float)
    individual_te.loc[df.index[valid_mask]] = individual_te_valid

    return {
        'ate': float(ate_local),
        'ps_scores': e_hat,
        'individual_te': individual_te,
        'valid_mask': valid_mask,
    }

# Point estimate using cross-fitted DR.
weight_col = 'FINAL_W'
analytic_df[weight_col] = W
point_fit = fit_cross_fitted_aipw(
    analytic_df,
    confounder_cols=confounder_cols,
    treatment_col='T_High_Centrality',
    outcome_col='W5_Cognitive_Composite',
    weight_col=weight_col,
    n_splits=5,
    random_state=42,
)

weighted_ATE = point_fit['ate']
ps_scores = point_fit['ps_scores']
individual_TE = point_fit['individual_te']
valid_mask = point_fit['valid_mask']
T_local = analytic_df['T_High_Centrality'].astype(int).to_numpy()

# Positivity diagnostics.
treated_ps = ps_scores[(T_local == 1) & valid_mask]
control_ps = ps_scores[(T_local == 0) & valid_mask]
frac_clipped = np.mean((ps_scores[valid_mask] <= 0.05) | (ps_scores[valid_mask] >= 0.95))
print("=== Positivity / Overlap Diagnostics (Cross-fitted PS) ===")
print(f"Valid cross-fitted rows: {valid_mask.sum()} / {len(valid_mask)}")
print(f"Treated PS range (5th, 50th, 95th): {np.quantile(treated_ps, [0.05, 0.5, 0.95])}")
print(f"Control PS range (5th, 50th, 95th): {np.quantile(control_ps, [0.05, 0.5, 0.95])}")
print(f"Fraction of clipped propensity scores: {frac_clipped:.3f}")

# Cluster-robust (approximate) influence-function SE.
weight_sum = W[valid_mask].sum()
influence = W[valid_mask] * (individual_TE.loc[analytic_df.index[valid_mask]] - weighted_ATE) / weight_sum
cluster = analytic_df.loc[valid_mask, 'CLUSTER2']
cluster_influence = influence.groupby(cluster).sum()
G = cluster_influence.shape[0]
se_ATE_if = np.sqrt((G / (G - 1)) * np.sum(cluster_influence.values ** 2))

# Cluster bootstrap with cross-fitted nuisance models.
rng = np.random.default_rng(42)
clusters = analytic_df['CLUSTER2'].dropna().unique()
B = 300
boot_ates = []

for _ in range(B):
    sampled_clusters = rng.choice(clusters, size=len(clusters), replace=True)
    boot_df = pd.concat(
        [analytic_df[analytic_df['CLUSTER2'] == g] for g in sampled_clusters],
        axis=0,
        ignore_index=True,
    )

    # Skip degenerate resamples with one treatment arm only.
    if boot_df['T_High_Centrality'].nunique() < 2:
        continue

    try:
        boot_fit = fit_cross_fitted_aipw(
            boot_df,
            confounder_cols=confounder_cols,
            treatment_col='T_High_Centrality',
            outcome_col='W5_Cognitive_Composite',
            weight_col=weight_col,
            n_splits=5,
            random_state=42,
        )
        boot_ates.append(boot_fit['ate'])
    except Exception:
        continue

boot_ates = np.asarray(boot_ates)
boot_n = len(boot_ates)

if boot_n >= 100:
    se_ATE_boot = boot_ates.std(ddof=1)
    ci_lower_boot, ci_upper_boot = np.quantile(boot_ates, [0.025, 0.975])
    p_boot = 2 * min(np.mean(boot_ates <= 0), np.mean(boot_ates >= 0))
else:
    se_ATE_boot = np.nan
    ci_lower_boot, ci_upper_boot = np.nan, np.nan
    p_boot = np.nan

# t-based inference with approximate IF SE (legacy-compatible display).
df_t = G - 1
t_stat = weighted_ATE / se_ATE_if
p_value_if = 2 * (1 - stats.t.cdf(abs(t_stat), df=df_t))
t_crit = stats.t.ppf(0.975, df=df_t)
ci_lower_if = weighted_ATE - t_crit * se_ATE_if
ci_upper_if = weighted_ATE + t_crit * se_ATE_if

print("\n=== Cross-Fitted Doubly Robust Results (AIPW) ===")
print(f"Sample Size (N, valid CF rows)       : {valid_mask.sum()}")
print(f"Number of clusters (PSU)             : {G}")
print(f"Weighted ATE                         : {weighted_ATE:.4f}")
print(f"Approx cluster IF SE                 : {se_ATE_if:.4f}")
print(f"95% CI (IF t, df={df_t})              : [{ci_lower_if:.4f}, {ci_upper_if:.4f}]")
print(f"P-value (IF t-test)                  : {p_value_if:.4f}")
print(f"Cluster bootstrap reps kept          : {boot_n} / {B}")
if boot_n >= 100:
    print(f"Cluster bootstrap SE                 : {se_ATE_boot:.4f}")
    print(f"95% CI (bootstrap percentile)        : [{ci_lower_boot:.4f}, {ci_upper_boot:.4f}]")
    print(f"P-value (bootstrap sign test)        : {p_boot:.4f}")
else:
    print("Bootstrap diagnostics unavailable: too few valid replicates.")

if not np.isnan(p_boot):
    if p_boot < 0.05:
        print("\nConclusion (bootstrap): Statistically significant effect at alpha=0.05.")
    else:
        print("\nConclusion (bootstrap): No statistically significant effect at alpha=0.05.")
else:
    if p_value_if < 0.05:
        print("\nConclusion (IF): Statistically significant effect at alpha=0.05.")
    else:
        print("\nConclusion (IF): No statistically significant effect at alpha=0.05.")

=== Positivity / Overlap Diagnostics (Cross-fitted PS) ===
Valid cross-fitted rows: 394 / 394
Treated PS range (5th, 50th, 95th): [0.0785227  0.57557553 0.92836166]
Control PS range (5th, 50th, 95th): [0.05841066 0.51967443 0.88041964]
Fraction of clipped propensity scores: 0.086

=== Cross-Fitted Doubly Robust Results (AIPW) ===
Sample Size (N, valid CF rows)       : 394
Number of clusters (PSU)             : 107
Weighted ATE                         : -0.2911
Approx cluster IF SE                 : 0.4742
95% CI (IF t, df=106)              : [-1.2313, 0.6491]
P-value (IF t-test)                  : 0.5407
Cluster bootstrap reps kept          : 300 / 300
Cluster bootstrap SE                 : 1.0687
95% CI (bootstrap percentile)        : [-2.3637, 1.9583]
P-value (bootstrap sign test)        : 0.6933

Conclusion (bootstrap): No statistically significant effect at alpha=0.05.


In [10]:
# Sensitivity analysis for unobserved confounding (bias-function / tipping-point)
# Idea: ATE_true = ATE_obs - gamma * delta_p
# gamma   = effect of latent confounder U on outcome (in Y units)
# delta_p = prevalence difference of U between treated and control

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Pull observed estimate + uncertainty from prior cell
ate_obs = float(weighted_ATE)
se_obs = float(se_ATE_if) if 'se_ATE_if' in globals() else np.nan

# Rebuild treatment/outcome/weights from current analytic_df
T_sens = analytic_df['T_High_Centrality'].astype(int).to_numpy()
Y_sens = analytic_df['W5_Cognitive_Composite'].to_numpy()
W_sens = analytic_df['FINAL_W'].to_numpy() if 'FINAL_W' in analytic_df.columns else W.to_numpy()

def wmean(x, w):
    return np.sum(w * x) / np.sum(w)

# Approximate benchmark strengths from observed covariates:
# 1) delta_p benchmark from max weighted |SMD|
delta_p_bench = float(smd_df['abs_smd'].max()) if 'smd_df' in globals() and len(smd_df) > 0 else np.nan

# 2) gamma benchmark from strongest weighted 1-SD association with Y among observed X
def weighted_sd(x, w):
    mu = wmean(x, w)
    return np.sqrt(np.sum(w * (x - mu) ** 2) / np.sum(w))

gamma_candidates = []
for c in confounder_cols:
    x = pd.to_numeric(analytic_df[c], errors='coerce').to_numpy()
    m = np.isfinite(x) & np.isfinite(Y_sens) & np.isfinite(W_sens)
    if m.sum() < 30:
        continue
    x_m = x[m]
    y_m = Y_sens[m]
    w_m = W_sens[m]
    x_sd = weighted_sd(x_m, w_m)
    if x_sd == 0 or not np.isfinite(x_sd):
        continue
    x_std = (x_m - wmean(x_m, w_m)) / x_sd
    lr = LinearRegression()
    lr.fit(x_std.reshape(-1, 1), y_m, sample_weight=w_m)
    gamma_candidates.append(abs(float(lr.coef_[0])))

gamma_bench = np.nan if len(gamma_candidates) == 0 else float(np.max(gamma_candidates))

print("=== Unobserved Confounding Sensitivity (Bias Function) ===")
print(f"Observed ATE: {ate_obs:.4f}")
if np.isfinite(se_obs):
    print(f"Approx IF SE: {se_obs:.4f}")
print(f"Benchmark delta_p (max observed |SMD|): {delta_p_bench:.4f}")
print(f"Benchmark gamma (max observed 1-SD effect on Y): {gamma_bench:.4f}")

# Benchmark implied bias and adjusted ATE
if np.isfinite(delta_p_bench) and np.isfinite(gamma_bench):
    bias_bench = gamma_bench * delta_p_bench
    ate_adj_bench = ate_obs - np.sign(ate_obs) * bias_bench
    print(f"Benchmark bias magnitude gamma*delta_p: {bias_bench:.4f}")
    print(f"Benchmark adjusted ATE (worst direction): {ate_adj_bench:.4f}")

# Tipping-point grid
gamma_grid = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.75, 1.00])
delta_grid = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50])

rows = []
for g in gamma_grid:
    for d in delta_grid:
        bias_mag = g * d
        ate_worst = ate_obs - np.sign(ate_obs) * bias_mag
        if np.isfinite(se_obs):
            z_worst = abs(ate_worst) / se_obs if se_obs > 0 else np.nan
            robust_to_zero_95 = abs(ate_worst) > 1.96 * se_obs
        else:
            z_worst = np.nan
            robust_to_zero_95 = np.nan
        rows.append({
            "gamma": g,
            "delta_p": d,
            "bias_mag": bias_mag,
            "ate_worst_direction": ate_worst,
            "z_worst": z_worst,
            "robust_to_zero_95": robust_to_zero_95
        })

sens_df = pd.DataFrame(rows).sort_values(["bias_mag", "gamma", "delta_p"]).reset_index(drop=True)

# Minimum bias needed to move point estimate to zero
min_bias_to_zero = abs(ate_obs)
print(f"\nMinimum |gamma * delta_p| needed to move point estimate to 0: {min_bias_to_zero:.4f}")

# First combinations that can explain away point estimate
explain_away = sens_df[sens_df["bias_mag"] >= min_bias_to_zero].copy()
if len(explain_away) > 0:
    print("\nSmallest (gamma, delta_p) pairs that can explain away point estimate:")
    print(explain_away.head(10).to_string(index=False))
else:
    print("\nNo tested (gamma, delta_p) pair in the grid explains away the point estimate.")

print("\nPreview of sensitivity table:")
print(sens_df.head(12).to_string(index=False))

=== Unobserved Confounding Sensitivity (Bias Function) ===
Observed ATE: -0.2911
Approx IF SE: 0.4742
Benchmark delta_p (max observed |SMD|): 0.3472
Benchmark gamma (max observed 1-SD effect on Y): 0.2356
Benchmark bias magnitude gamma*delta_p: 0.0818
Benchmark adjusted ATE (worst direction): -0.2093

Minimum |gamma * delta_p| needed to move point estimate to 0: 0.2911

Smallest (gamma, delta_p) pairs that can explain away point estimate:
 gamma  delta_p  bias_mag  ate_worst_direction  z_worst  robust_to_zero_95
  1.00      0.3     0.300             0.008901 0.018768              False
  0.75      0.4     0.300             0.008901 0.018768              False
  0.75      0.5     0.375             0.083901 0.176913              False
  1.00      0.4     0.400             0.108901 0.229628              False
  1.00      0.5     0.500             0.208901 0.440488              False

Preview of sensitivity table:
 gamma  delta_p  bias_mag  ate_worst_direction  z_worst  robust_to_zero_95
 